<h4>Implement a simple map-reduce job that builds an inverted index on the set of input</h4>

In [ ]:
pip install mrjob

In [38]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 2.2 MB/s  0:00:04m 2.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 2.0 MB/s  0:00:08 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [69]:
pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.5 MB/s  0:00:042.5 MB/s eta 0:00:01:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.3 MB/s  0:00:02m 2.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.4 MB/s  0:00:002.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 2.5 MB/s  0:00:022.5 MB/s eta 0:00:01:01
Using cached pyparsing-3.3.3-py3-none-any.whl (126 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib] 6/7 [matplotlib]ow]
Note: you may need to restart the kernel to use updated packages.


In [48]:
#checking availability of required libraries 
!pip show mrjob

Name: mrjob
Version: 0.7.4
Summary: Python MapReduce framework
Home-page: http://github.com/Yelp/mrjob
Author: David Marin
Author-email: dm@davidmarin.org
License: Apache
Location: /home/divya/anaconda3/envs/mapreduce/lib/python3.12/site-packages
Requires: PyYAML
Required-by: 


<p>Importing the necessary libraries</p>

<ol>
    <li>create a class called WordCount</li>
    <li>create a mapper function inside the class</li>
    <li>create a reducer function inside the class</li>
</ol>

In [24]:
%%writefile MapReduce_InvertedIndex.py
from mrjob.job import MRJob
import pandas as pd
import matplotlib.pyplot as plt
import re

# Regular expression to get clean words (removing punctuation)
WORD_RE = re.compile(r"[\w']+")

class InvertedIndexAndFrequency(MRJob):
    
    def mapper_init(self):
        # Initialize a line counter for this split/file mapper instance
        self.line_number = 0

    def mapper(self, _, line):
        self.line_number += 1
        # break down line into individual clean words 
        for word in WORD_RE.findall(line):
            word_lower = word.lower()
            # Yield word as key, and a dictionary containing line occurrence and a count of 1
            yield (word_lower, {"line": self.line_number, "count": 1})
              
    def reducer(self, word, values):
        total_count = 0
        lines_found = set() #datatype store an unordered collection of unique items
        
        # add counts and unique line numbers for each word
        for val in values:
            total_count += val["count"]
            lines_found.add(val["line"])
            
        # Yield the word mapping to its frequency and sorted list of lines
        yield (word, {"count": total_count, "lines": sorted(list(lines_found))})

if __name__ == '__main__':
    # Simulating data execution using 'inputtext1.txt'
    mr_job = InvertedIndexAndFrequency(args=['inputtext1.txt'])
    results = []
    
    with mr_job.make_runner() as runner:
        runner.run()
        for key, value in mr_job.parse_output(runner.cat_output()):
            results.append({
                'word': key, 
                'count': value['count'], 
                'lines': value['lines']
            })
            
    #print the sample inverted index dictionary items
    print("Inverted Index Sample (First 5):")
    print(results[:5])
    
    #build Pandas DataFrame
    wordCount_df = pd.DataFrame(results)
    
    print("\nPandas DataFrame Head (Inverted Index Mapping):")
    print(wordCount_df[['word', 'lines', 'count']].head())
    
    # display chart of frequently occuring words
    top_words = wordCount_df.sort_values(by="count", ascending=False).head(10)
    print("\nTop 10 Most Frequent Words:")
    print(top_words[['word', 'count']])
    
    #plot
    top_words.plot(x="word", y="count", kind="bar", color="green")
    plt.title("Top 10 Most Frequent Words")
    plt.ylabel("Occurrences")
    plt.savefig("word_count_chart.png")
    print("\nPlot saved successfully as 'word_count_chart.png'")


Overwriting MapReduce_InvertedIndex.py


In [25]:
!python MapReduce_InvertedIndex.py inputtext1.txt

No configs specified for inline runner
Inverted Index Sample (First 5):
[{'word': 'to', 'count': 2, 'lines': [1]}, {'word': 'troubled', 'count': 1, 'lines': [1]}, {'word': 'up', 'count': 1, 'lines': [1]}, {'word': 'violent', 'count': 1, 'lines': [1]}, {'word': 'is', 'count': 2, 'lines': [1]}]

Pandas DataFrame Head (Inverted Index Mapping):
       word lines  count
0        to   [1]      2
1  troubled   [1]      1
2        up   [1]      1
3   violent   [1]      1
4        is   [1]      2

Top 10 Most Frequent Words:
       word  count
27        a      5
49       in      5
6    liliom      4
52      the      3
44      his      3
0        to      2
4        is      2
5     julie      2
8   robbery      2
30     play      2

Plot saved successfully as 'word_count_chart.png'
